In [68]:
"""
Graph RAG / Knowledge Graph RAG -- Production-Grade Pipeline
=============================================================
Architecture   : THREE retrieval modes working in concert:
                 (1) Vector Search         -- semantic similarity on chunk embeddings
                 (2) Graph Traversal       -- structured Cypher queries on entity graph
                 (3) Hybrid RAG            -- vector + full-text + graph combined
LLM            : Azure OpenAI  (AzureChatOpenAI -- GPT-4o)
Embeddings     : HuggingFace sentence-transformers/all-MiniLM-L6-v2  (local, no API key)
Graph Database : Neo4j  (free AuraDB cloud instance OR local Docker)
Graph Builder  : LLMGraphTransformer  (langchain-experimental)
Vector Index   : Neo4jVector  (langchain-neo4j official partner package)
Data Source    : Three publicly available research PDFs from arXiv

Reference PDFs:
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf

    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf

    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

=============================================================================
Core Concept -- WHY Graph RAG vs. Standard / Semantic / Multi-Vector RAG
=============================================================================

All previous RAG architectures in this series (Standard, PDR, MVR, Semantic)
operate on the SAME fundamental model: embed text chunks, retrieve by cosine
similarity, feed chunks to LLM. This model has a hard structural limitation:

    It can only retrieve WHAT IS WRITTEN in a specific chunk.
    It CANNOT answer questions that require TRAVERSING RELATIONSHIPS
    between entities that appear in DIFFERENT parts of the document.

Example failure of standard RAG:
    Q: "Which papers cite both the Transformer architecture and BERT?"
    Standard RAG: retrieves chunks about Transformers OR chunks about BERT,
    but cannot REASON about the relationship between them across documents.

Graph RAG solves this by doing two things simultaneously:
    1. It stores the TEXT in a vector index (for semantic search).
    2. It extracts ENTITIES and RELATIONSHIPS from the text into a property
       graph (for structured traversal).

At retrieval time it can ask:
    "Find all Concept nodes connected to both 'Transformer' and 'BERT'
     via INFLUENCES or BUILDS_ON relationships."

This multi-hop traversal is impossible with any embedding-only approach.

=============================================================================
Graph RAG Architecture: Three Retrieval Modes
=============================================================================

MODE 1 -- VECTOR RETRIEVAL (semantic baseline inside Neo4j)
    Chunks stored as (:Chunk) nodes with 384-dim embedding properties.
    Neo4j vector index enables cosine similarity search natively.
    Same semantic matching as ChromaDB but INSIDE the graph database,
    enabling seamless combination with graph traversal in a single query.

MODE 2 -- GRAPH TRAVERSAL via Text2Cypher (GraphCypherQAChain)
    LLMGraphTransformer extracts (:Entity)-[:RELATIONSHIP]->(:Entity) triples.
    User question is translated to Cypher by Azure OpenAI.
    Cypher executes against the live Neo4j graph.
    Returns structured data: entity names, relationship types, properties.
    Best for: "Who/What/How are X and Y related?", multi-hop reasoning.

MODE 3 -- HYBRID RETRIEVAL (vector + keyword + graph context)
    Neo4jVector with search_type="hybrid" combines:
        - Vector similarity search (semantic)
        - Full-text BM25 keyword search (lexical)
        - Custom Cypher retrieval_query to enrich each chunk with its
          connected entities and relationships from the graph.
    This is the most powerful mode: every retrieved chunk is augmented
    with the structured knowledge graph context of its entities.

=============================================================================
Pipeline Flow
=============================================================================

PDF --> PyPDFLoader --> pages
    |
    v
TokenTextSplitter (512 tokens, overlap 24)
    --> (:Chunk) nodes + text embeddings --> Neo4j vector index
    |
    v
LLMGraphTransformer (Azure OpenAI, batch async)
    --> (:Entity {name, type, description})-[:RELATION {description}]->(:Entity)
    --> graph.add_graph_documents(include_source=True)
    --> Links: (:Chunk)-[:MENTIONS]->(:Entity)
    |
    v
At Query Time:
    Mode 1: embed query --> Neo4j vector index --> similar chunks --> LLM
    Mode 2: question --> Text2Cypher (LLM) --> Cypher --> graph results --> LLM
    Mode 3: embed + BM25 + entity enrichment --> augmented chunks --> LLM


"""

'\nGraph RAG / Knowledge Graph RAG -- Production-Grade Pipeline\n=============================================================\nArchitecture   : THREE retrieval modes working in concert:\n                 (1) Vector Search         -- semantic similarity on chunk embeddings\n                 (2) Graph Traversal       -- structured Cypher queries on entity graph\n                 (3) Hybrid RAG            -- vector + full-text + graph combined\nLLM            : Azure OpenAI  (AzureChatOpenAI -- GPT-4o)\nEmbeddings     : HuggingFace sentence-transformers/all-MiniLM-L6-v2  (local, no API key)\nGraph Database : Neo4j  (free AuraDB cloud instance OR local Docker)\nGraph Builder  : LLMGraphTransformer  (langchain-experimental)\nVector Index   : Neo4jVector  (langchain-neo4j official partner package)\nData Source    : Three publicly available research PDFs from arXiv\n\nReference PDFs:\n    1. "Attention Is All You Need" -- Vaswani et al., 2017\n       https://arxiv.org/pdf/1706.03762.pdf\n\n 

In [69]:

!pip install langchain-neo4j

In [70]:
import os
import sys
import time
import logging
import asyncio
import urllib.request
from pathlib import Path
from typing import List, Dict, Tuple, Optional


from langchain_core.documents import Document 
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.graph_transformers import  LLMGraphTransformer


from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import AzureChatOpenAI

from langchain_core.prompts import ChatPromptTemplate ,PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [71]:
from langchain_text_splitters import TokenTextSplitter

In [72]:
from langchain_neo4j import (
    Neo4jGraph,
    Neo4jVector,
    GraphCypherQAChain,
)

In [73]:
NEO4J_URI= "neo4j+s://3825951c.databases.neo4j.io"
NEO4J_USERNAME="3825951c"
NEO4J_PASSWORD="Jc_SwyTY17g9C7j4JDodRaeCXyq2MnzvoOnTX1-7XEM"
NEO4J_DATABASE="3825951c"
AURA_INSTANCEID="3825951c"
AURA_INSTANCENAME="Free instance"


In [74]:
AI_FOUNDRY_PROJECT_ENDPOINT= "https://dhanush-ai507.cognitiveservices.azure.com/"
AI_FOUNDRY_DEPLOYMENT_NAME= "gpt-4.1"
AI_FOUNDRY_API_VERSION= "2024-12-01-preview"
AI_FOUNDRY_API_KEY= ""


In [75]:
# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("graph_rag")


In [93]:

# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class GraphRAGConfig:
    """
    Centralized configuration for the Graph RAG pipeline.

    Neo4j Setup Options:
        OPTION A (Recommended for testing): Neo4j AuraDB Free Tier
            1. Go to https://neo4j.com/cloud/platform/aura-graph-database/
            2. Create a free instance (no credit card required)
            3. Copy the connection URI, username, and password
            4. Set NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD env vars

        OPTION B (Local Docker):
            docker run --name neo4j -p 7474:7474 -p 7687:7687 \
                -e NEO4J_AUTH=neo4j/password \
                -e NEO4J_PLUGINS='["apoc","graph-data-science"]' \
                neo4j:5.15

        OPTION C (Local Desktop): Download from https://neo4j.com/download/

    LLMGraphTransformer Schema:
        Defining ALLOWED_NODES and ALLOWED_RELATIONSHIPS constrains the LLM
        to extract only the entity and relationship types relevant to the domain.
        Unconstrained extraction produces noisy, inconsistent graphs.
        For research papers, we define: Concept, Method, Model, Author, Dataset,
        Metric as node types and: PROPOSES, BUILDS_ON, EVALUATES_ON, USES,
        OUTPERFORMS, INTRODUCES, CITES as relationship types.

    TokenTextSplitter vs RecursiveCharacterTextSplitter:
        We use TokenTextSplitter for graph construction because:
        1. LLMGraphTransformer context is token-bounded (GPT-4o: 128K tokens).
           Token splitting guarantees no chunk exceeds the model's context window.
        2. Graph extraction quality is consistent across equal-token chunks.
        3. The 512-token / 24-token-overlap configuration matches the original
           Neo4j GraphRAG paper benchmark setup for research document ingestion.
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",    "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers", "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",  "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # --- Text splitting (token-based for graph extraction) ----------------
    CHUNK_SIZE_TOKENS: int    = 512
    CHUNK_OVERLAP_TOKENS: int = 24

    # --- HuggingFace Embeddings (local, no API key) -----------------------
    EMBEDDING_MODEL: str  = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str = "cpu"              # "cuda" for GPU

    # --- Neo4j Connection --------------------------------------------------
    # Prefer environment variables; fall back to values set in notebook cells.
    NEO4J_URI: str = os.environ.get("NEO4J_URI", NEO4J_URI)
    NEO4J_USERNAME: str = os.environ.get("NEO4J_USERNAME", NEO4J_USERNAME)
    NEO4J_PASSWORD: str = os.environ.get("NEO4J_PASSWORD", NEO4J_PASSWORD)
    NEO4J_DATABASE: str = os.environ.get("NEO4J_DATABASE", NEO4J_DATABASE)

    # --- Neo4j Index Names ------------------------------------------------
    VECTOR_INDEX_NAME:   str = "research_chunks"
    KEYWORD_INDEX_NAME:  str = "research_keyword"
    EMBEDDING_DIMENSION: int = 384              # all-MiniLM-L6-v2 output dim

    # --- LLMGraphTransformer Schema ---------------------------------------
    # Constraining allowed types dramatically improves graph consistency.
    # The LLM will only extract these specific node and relationship types.
    ALLOWED_NODES: List[str] = [
        "Concept", "Method", "Model", "Architecture",
        "Author", "Dataset", "Metric", "Paper",
    ]
    ALLOWED_RELATIONSHIPS: List[str] = [
        "PROPOSES", "BUILDS_ON", "EVALUATES_ON", "USES",
        "OUTPERFORMS", "INTRODUCES", "CITES", "APPLIES_TO",
        "COMPARES_WITH", "IMPROVES",
    ]

    # LLMGraphTransformer async batch concurrency (parallel Azure API calls)
    GRAPH_EXTRACTION_CONCURRENCY: int = 5

    # --- Azure OpenAI LLM -------------------------------------------------
    # AZURE_OPENAI_API_KEY:    str  = os.environ.get("AZURE_OPENAI_API_KEY",    "")
    # AZURE_OPENAI_ENDPOINT:   str  = os.environ.get("AZURE_OPENAI_ENDPOINT",   "")
    # AZURE_OPENAI_DEPLOYMENT: str  = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
    # AZURE_OPENAI_API_VERSION: str = os.environ.get("AZURE_OPENAI_API_VERSION","2024-02-15-preview")
    LLM_TEMPERATURE: float        = 0.0
    LLM_MAX_TOKENS: int           = 1024

    # --- Retrieval --------------------------------------------------------
    VECTOR_K: int = 4             # top-k chunks for vector retrieval
    HYBRID_K: int = 4             # top-k chunks for hybrid retrieval
    CYPHER_K: int = 3             # top-k graph results for Cypher retrieval

    # --- RAG Answer Prompt (Modes 1 and 3) --------------------------------
    RAG_PROMPT: str = """You are a precise technical research assistant.
Answer the question using ONLY the information provided in the context below.
If the answer is not present in the context, respond with:
"The provided documents do not contain enough information to answer this question."

Context (retrieved from research papers with knowledge graph enrichment):
{context}

Question: {question}

Provide a technically accurate, well-structured answer:"""

    # --- Custom Cypher Retrieval Query for Hybrid Mode --------------------
    # This query enriches each retrieved Chunk with its connected entities
    # and relationships from the knowledge graph. The result is that every
    # chunk the LLM receives is annotated with structured graph context.
    HYBRID_RETRIEVAL_QUERY: str = """
MATCH (node)-[:MENTIONS]->(entity)
WITH node, score, collect(entity.id + " [" + labels(entity)[0] + "]") AS entities
RETURN node.text AS text,
       score,
       node {
           .source, .paper_name, .page,
           entities: entities
       } AS metadata
"""

In [77]:


# ===========================================================================
# SECTION 2 -- PDF DOWNLOADER
# ===========================================================================

def download_pdfs(config: GraphRAGConfig) -> List[Path]:
    """
    Download research PDFs from arXiv to local filesystem with caching.

    Args:
        config: GraphRAGConfig with PDF_SOURCES and PDF_DOWNLOAD_DIR.

    Returns:
        List of Path objects for downloaded PDFs.

    Raises:
        RuntimeError: If download fails or produces unexpectedly small file.
    """
    download_dir = Path(config.PDF_DOWNLOAD_DIR)
    download_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []

    for name, url in config.PDF_SOURCES:
        local_path = download_dir / f"{name}.pdf"

        if local_path.exists() and local_path.stat().st_size > 10_000:
            logger.info("Cached: %s (%.1f KB)", local_path.name, local_path.stat().st_size / 1024)
            paths.append(local_path)
            continue

        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0 (RAG-Research-Pipeline/1.0)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()

            if len(data) < 10_000:
                raise RuntimeError(f"File too small ({len(data)} bytes): {url}")

            local_path.write_bytes(data)
            logger.info("Saved: %s (%.1f KB)", local_path.name, len(data) / 1024)
            paths.append(local_path)
            time.sleep(1.0)

        except Exception as exc:
            raise RuntimeError(
                f"Cannot download '{url}'. "
                f"Manually place the file at '{local_path}'."
            ) from exc

    return paths



In [78]:

# ===========================================================================
# SECTION 3 -- PDF LOADING AND TOKEN SPLITTING
# ===========================================================================

def load_and_split_documents(
    pdf_paths: List[Path],
    config: GraphRAGConfig,
) -> List[Document]:
    """
    Load PDF pages via PyPDFLoader and split into fixed-token chunks.

    TokenTextSplitter uses tiktoken (cl100k_base encoding, same as GPT-4)
    to split text into exactly CHUNK_SIZE_TOKENS tokens per chunk. This is
    the correct splitter when downstream processing is LLM-based (as in
    LLMGraphTransformer) because:
        1. LLM context windows are bounded in TOKENS not characters.
        2. A 512-token chunk for gpt-4o costs approximately 0.0025 USD input
           per call (at $5/1M tokens), which is predictable and auditable.
        3. Token-based chunks produce consistent entity extraction quality
           because the LLM always receives roughly the same amount of text.

    The 24-token overlap (roughly 2 sentences) preserves cross-boundary
    entities that a zero-overlap split would cut in half.

    Args:
        pdf_paths: Downloaded PDF Path objects.
        config   : GraphRAGConfig with splitting parameters.

    Returns:
        List of tokenized Documents, each annotated with paper_name metadata.
    """
    splitter = TokenTextSplitter(
        chunk_size=config.CHUNK_SIZE_TOKENS,
        chunk_overlap=config.CHUNK_OVERLAP_TOKENS,
    )

    all_chunks: List[Document] = []

    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        logger.info("Loading: %s", pdf_path.name)

        try:
            pages = PyPDFLoader(str(pdf_path)).load()
        except Exception as exc:
            raise RuntimeError(f"Failed to load '{pdf_path}': {exc}") from exc

        for page in pages:
            page.metadata["source"]     = pdf_path.name
            page.metadata["paper_name"] = paper_name

        chunks = splitter.split_documents(pages)
        logger.info(
            "  %s --> %d pages --> %d token chunks",
            pdf_path.name, len(pages), len(chunks),
        )
        all_chunks.extend(chunks)

    logger.info("Total chunks across all PDFs: %d", len(all_chunks))
    return all_chunks


In [79]:

# ===========================================================================
# SECTION 4 -- AZURE OPENAI LLM INITIALIZATION
# ===========================================================================

def build_azure_llm(config: GraphRAGConfig) -> AzureChatOpenAI:
    """
    Initialize and validate the Azure OpenAI LLM.

    Used for TWO distinct purposes in this pipeline:
        1. LLMGraphTransformer entity/relationship extraction at ingestion time.
        2. Answer generation from retrieved context at query time.

    Both purposes benefit from temperature=0 for determinism. The graph
    extraction step uses function_calling (structured output) internally in
    LLMGraphTransformer, which is fully supported by Azure OpenAI GPT-4o.

    Args:
        config: GraphRAGConfig with Azure credentials.

    Returns:
        Initialized AzureChatOpenAI instance.

    Raises:
        EnvironmentError: If required Azure credentials are not set.
    """
    AI_FOUNDRY_PROJECT_ENDPOINT= "https://dhanush-ai507.cognitiveservices.azure.com/"
    AI_FOUNDRY_DEPLOYMENT_NAME= "gpt-4.1"
    AI_FOUNDRY_API_VERSION= "2024-12-01-preview"
    AI_FOUNDRY_API_KEY= ""

    missing = [
        name for name, val in [
            ("AZURE_OPENAI_API_KEY",  AI_FOUNDRY_API_KEY),
            ("AZURE_OPENAI_ENDPOINT", AI_FOUNDRY_PROJECT_ENDPOINT),
        ] if not val
    ]
    if missing:
        raise EnvironmentError(
            f"Missing required environment variables: {missing}\n"
            "  export AZURE_OPENAI_API_KEY='your-key'\n"
            "  export AZURE_OPENAI_ENDPOINT='https://your-resource.openai.azure.com/'\n"
            "  export AZURE_OPENAI_DEPLOYMENT='gpt-4o'\n"
            "  export AZURE_OPENAI_API_VERSION='2024-02-15-preview'"
        )

    logger.info(
        "Azure OpenAI: deployment='%s'  api_version='%s'",
        AI_FOUNDRY_DEPLOYMENT_NAME, AI_FOUNDRY_API_VERSION,
    )

    return AzureChatOpenAI(
        azure_deployment=AI_FOUNDRY_DEPLOYMENT_NAME,
        azure_endpoint=AI_FOUNDRY_PROJECT_ENDPOINT,
        api_key=AI_FOUNDRY_API_KEY,
        api_version=AI_FOUNDRY_API_VERSION,
        temperature=config.LLM_TEMPERATURE,
        max_tokens=config.LLM_MAX_TOKENS,
    )



In [80]:


# ===========================================================================
# SECTION 5 -- EMBEDDING MODEL INITIALIZATION
# ===========================================================================

def build_embeddings(config: GraphRAGConfig) -> HuggingFaceEmbeddings:
    """
    Initialize the HuggingFace sentence-transformers embedding model.

    all-MiniLM-L6-v2 produces 384-dimensional unit-normalized vectors and
    runs fully locally. EMBEDDING_DIMENSION=384 must match this output
    dimension when creating the Neo4j vector index.

    Args:
        config: GraphRAGConfig with EMBEDDING_MODEL and EMBEDDING_DEVICE.

    Returns:
        Initialized HuggingFaceEmbeddings instance.
    """
    logger.info("Embedding model: %s (device=%s)", config.EMBEDDING_MODEL, config.EMBEDDING_DEVICE)
    return HuggingFaceEmbeddings(
        model_name=config.EMBEDDING_MODEL,
        model_kwargs={"device": config.EMBEDDING_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )

In [99]:

# ===========================================================================
# SECTION 6 -- NEO4J GRAPH CONNECTION
# ===========================================================================

def connect_to_neo4j(config: GraphRAGConfig) -> Neo4jGraph:
    """
    Establish and validate connection to Neo4j.

    Neo4jGraph is a thin wrapper around the Neo4j Python driver that
    exposes .query() for executing Cypher statements and .refresh_schema()
    to pull the current graph schema. GraphCypherQAChain uses refresh_schema()
    to give the LLM an up-to-date schema description for Text2Cypher generation.

    refresh_schema=True on initialization automatically fetches the schema,
    which is needed before GraphCypherQAChain can generate accurate Cypher.

    Args:
        config: GraphRAGConfig with NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD.

    Returns:
        Connected Neo4jGraph instance.

    Raises:
        RuntimeError: If Neo4j connection cannot be established.
    """
    base_uri = config.NEO4J_URI.strip()
    candidate_uris = [base_uri]
    if base_uri.startswith("neo4j+s://"):
        candidate_uris.append(base_uri.replace("neo4j+s://", "bolt+s://", 1))
        candidate_uris.append(base_uri.replace("neo4j+s://", "neo4j://", 1))
    elif base_uri.startswith("neo4j://"):
        candidate_uris.append(base_uri.replace("neo4j://", "neo4j+s://", 1))
        candidate_uris.append(base_uri.replace("neo4j://", "bolt+s://", 1))

    # Preserve order while removing duplicates.
    deduped_uris: List[str] = []
    for uri in candidate_uris:
        if uri not in deduped_uris:
            deduped_uris.append(uri)

    candidate_dbs = [config.NEO4J_DATABASE, "neo4j", None]
    deduped_dbs: List[Optional[str]] = []
    for db in candidate_dbs:
        if db not in deduped_dbs:
            deduped_dbs.append(db)

    logger.info("Connecting to Neo4j. URIs=%s | DBs=%s", deduped_uris, deduped_dbs)

    errors: List[str] = []
    for uri in deduped_uris:
        for db in deduped_dbs:
            try:
                graph = Neo4jGraph(
                    url=uri,
                    username=config.NEO4J_USERNAME,
                    password=config.NEO4J_PASSWORD,
                    database=db,
                    refresh_schema=True,
                )
                result = graph.query("RETURN 1 AS ping")
                assert result[0]["ping"] == 1
                logger.info("Neo4j connection established at %s (db=%s).", uri, db)
                return graph
            except Exception as exc:
                errors.append(f"uri={uri} db={db}: {exc}")

    raise RuntimeError(
        "Cannot connect to Neo4j using any tested URI/database combination. "
        f"Tried {len(deduped_uris) * len(deduped_dbs)} combinations. "
        "Verify Aura instance status, allowlist/network access, and credentials.\n"
        + "\n".join(errors[-3:])
    )



In [82]:

# ===========================================================================
# SECTION 7 -- GRAPH CONSTRUCTION: LLMGraphTransformer
# This is the core distinguishing step of Graph RAG. It converts unstructured
# text into a structured knowledge graph of entities and relationships.
# ===========================================================================

async def build_knowledge_graph_async(
    chunks: List[Document],
    llm: AzureChatOpenAI,
    graph: Neo4jGraph,
    config: GraphRAGConfig,
) -> None:
    """
    Extract entities and relationships from document chunks and store in Neo4j.

    LLMGraphTransformer uses Azure OpenAI's function_calling / structured_output
    mode to extract information as a list of GraphDocument objects. Each
    GraphDocument contains:
        - nodes     : List[Node]         -- entities with id, type, properties
        - relationships: List[Relationship] -- (source)-[type]->(target) triples
        - source    : Document           -- the originating text chunk

    The function_calling mode (default when the LLM supports it) is strictly
    superior to prompt_based mode because:
        1. The output is guaranteed to be valid JSON matching the schema.
        2. It supports extracting node and relationship PROPERTIES (descriptions).
        3. It does not require few-shot examples in the prompt.
        4. It produces isolated nodes (entities mentioned but not in a relation).

    graph.add_graph_documents() with include_source=True stores:
        - (:Entity {id, description}) nodes for each extracted entity
        - [:RELATIONSHIP {description}] edges between entities
        - (:Chunk {text, source, paper_name, page}) nodes for each source chunk
        - (:Chunk)-[:MENTIONS]->(:Entity) relationships linking text to entities

    The MENTIONS relationship is the critical bridge: it allows hybrid retrieval
    to take a vector-retrieved chunk and immediately ask "what entities does
    this chunk mention?" in the same Cypher query.

    We use aconvert_to_graph_documents (async) with gather() for parallel
    processing of GRAPH_EXTRACTION_CONCURRENCY chunks at a time. This reduces
    total extraction time from O(N * avg_latency) to O(N/C * avg_latency)
    where C is the concurrency level.

    Args:
        chunks  : Tokenized Document chunks from all PDFs.
        llm     : AzureChatOpenAI for entity extraction (function calling mode).
        graph   : Connected Neo4jGraph instance.
        config  : GraphRAGConfig with schema and concurrency settings.
    """
    logger.info(
        "Initializing LLMGraphTransformer with %d allowed node types "
        "and %d allowed relationship types ...",
        len(config.ALLOWED_NODES), len(config.ALLOWED_RELATIONSHIPS),
    )

    transformer = LLMGraphTransformer(
        llm=llm,
        allowed_nodes=config.ALLOWED_NODES,
        allowed_relationships=config.ALLOWED_RELATIONSHIPS,
        node_properties=["description"],            # extract description for each entity
        relationship_properties=["description"],    # extract description for each relation
    )

    total     = len(chunks)
    batch_sz  = config.GRAPH_EXTRACTION_CONCURRENCY
    processed = 0

    logger.info(
        "Extracting graph from %d chunks (async concurrency=%d) ...",
        total, batch_sz,
    )

    # Process chunks in concurrent batches to respect Azure rate limits
    # while maximizing throughput. Each batch issues batch_sz parallel calls.
    for batch_start in range(0, total, batch_sz):
        batch = chunks[batch_start : batch_start + batch_sz]

        # aconvert_to_graph_documents is the async version -- returns List[GraphDocument]
        graph_docs = await transformer.aconvert_to_graph_documents(batch)

        # Persist to Neo4j: store entities, relationships, and source chunk nodes
        # include_source=True creates (:Chunk) nodes linked via [:MENTIONS] to entities
        # baseEntityLabel=True adds a generic :__Entity__ label to all nodes,
        # enabling efficient full-graph scans without knowing specific labels
        graph.add_graph_documents(
            graph_docs,
            baseEntityLabel=True,
            include_source=True,
        )

        processed += len(batch)
        logger.info(
            "  Graph extraction progress: %d / %d chunks  "
            "(%.0f%%)",
            processed, total, 100 * processed / total,
        )

    # Refresh schema after ingestion so GraphCypherQAChain has an accurate
    # schema description for Text2Cypher generation
    graph.refresh_schema()

    # Report graph statistics
    node_count = graph.query("MATCH (n) RETURN count(n) AS cnt")[0]["cnt"]
    edge_count = graph.query("MATCH ()-[r]->() RETURN count(r) AS cnt")[0]["cnt"]
    entity_count = graph.query(
        "MATCH (n:__Entity__) RETURN count(n) AS cnt"
    )[0]["cnt"]

    logger.info(
        "Knowledge graph built. Nodes: %d  Edges: %d  Entities: %d",
        node_count, edge_count, entity_count,
    )


def build_knowledge_graph(
    chunks: List[Document],
    llm: AzureChatOpenAI,
    graph: Neo4jGraph,
    config: GraphRAGConfig,
) -> None:
    """
    Synchronous wrapper for build_knowledge_graph_async.

    Runs the async graph construction function in the current event loop.
    In Jupyter notebooks or async contexts, call build_knowledge_graph_async
    directly with await.

    Args:
        chunks  : Tokenized Document chunks.
        llm     : AzureChatOpenAI instance.
        graph   : Connected Neo4jGraph instance.
        config  : GraphRAGConfig.
    """
    asyncio.run(build_knowledge_graph_async(chunks, llm, graph, config))


In [95]:

# ===========================================================================
# SECTION 8 -- VECTOR INDEX CONSTRUCTION
# Creates a Neo4j native vector index on the (:Chunk) nodes for semantic search.
# ===========================================================================

def build_vector_index(
    chunks: List[Document],
    embeddings: HuggingFaceEmbeddings,
    graph: Neo4jGraph,
    config: GraphRAGConfig,
 ) -> Neo4jVector:
    """
    Build a Neo4j vector index on document chunks for semantic retrieval.

    Neo4jVector.from_documents():
        1. Creates (:Chunk) nodes if they don't exist (may reuse nodes created
           by add_graph_documents -- Neo4j merges on node identity).
        2. Embeds each chunk's text using the provided embedding model.
        3. Stores the embedding as a float array property on the (:Chunk) node.
        4. Creates a native vector index (ANN index) on this property using
           Neo4j's built-in vector index capability (available since Neo4j 5.11).

    The node_label="Chunk" and text_node_property="text" parameters must match
    the property names used by add_graph_documents(include_source=True), which
    stores source chunks as (:Chunk {text: ..., source: ..., paper_name: ...}).
    This ensures the vector index operates on the SAME nodes that contain the
    MENTIONS relationships to extracted entities.

    Args:
        chunks     : Document chunks to embed and index.
        embeddings : HuggingFace embedding model.
        graph      : Connected Neo4jGraph (provides connection details).
        config     : GraphRAGConfig with index names and dimensions.

    Returns:
        Neo4jVector instance ready for similarity_search.
    """
    logger.info(
        "Building Neo4j vector index '%s' on %d chunks ...",
        config.VECTOR_INDEX_NAME, len(chunks),
    )

    vectorstore = Neo4jVector.from_documents(
        documents=chunks,
        embedding=embeddings,
        url=config.NEO4J_URI,
        username=config.NEO4J_USERNAME,
        password=config.NEO4J_PASSWORD,
        database=config.NEO4J_DATABASE,
        index_name=config.VECTOR_INDEX_NAME,
        keyword_index_name=config.KEYWORD_INDEX_NAME,
        node_label="Chunk",
        text_node_property="text",
        embedding_node_property="embedding",
        search_type="hybrid",               # creates BOTH vector AND keyword indexes
        pre_delete_collection=False,        # do not drop existing index on re-run
    )

    logger.info(
        "Vector index '%s' built. Dimension: %d",
        config.VECTOR_INDEX_NAME, config.EMBEDDING_DIMENSION,
    )
    return vectorstore



In [96]:


# ===========================================================================
# SECTION 9 -- THREE RAG CHAIN BUILDERS
# ===========================================================================

def build_mode1_vector_chain(
    vectorstore: Neo4jVector,
    llm: AzureChatOpenAI,
    config: GraphRAGConfig,
):
    """
    Mode 1: Pure vector semantic search inside Neo4j.

    This is the semantic baseline: embed the query, find the k most similar
    (:Chunk) nodes by cosine distance in the Neo4j vector index, format their
    text as context, and send to the LLM for generation.

    This mode is equivalent to ChromaDB-backed RAG from the previous sessions
    but runs inside Neo4j, enabling the vector results to be trivially combined
    with graph traversal in the same database in Mode 3.

    Args:
        vectorstore : Neo4jVector with vector index.
        llm         : AzureChatOpenAI.
        config      : GraphRAGConfig.

    Returns:
        Tuple of (LCEL chain, retriever).
    """
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.VECTOR_K},
    )

    prompt = ChatPromptTemplate.from_template(config.RAG_PROMPT)

    def format_docs(docs: List[Document]) -> str:
        parts = []
        for i, doc in enumerate(docs, start=1):
            paper = doc.metadata.get("paper_name", "Unknown")
            page  = doc.metadata.get("page", "?")
            parts.append(f"[Chunk {i} | {paper} | Page {page}]\n{doc.page_content.strip()}")
        return ("\n\n" + "-" * 60 + "\n\n").join(parts)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )

    logger.info("Mode 1 (Vector) chain assembled.")
    return chain, retriever


def build_mode2_graph_cypher_chain(
    graph: Neo4jGraph,
    llm: AzureChatOpenAI,
    config: GraphRAGConfig,
) -> GraphCypherQAChain:
    """
    Mode 2: Graph traversal via Text2Cypher (GraphCypherQAChain).

    GraphCypherQAChain translates a natural language question into a Cypher
    query using the LLM (guided by the graph schema), executes the Cypher
    against Neo4j, then sends the structured results back to the LLM for
    natural language answer generation.

    This chain is the ONLY mode that can answer multi-hop relationship
    questions that span multiple documents, because it traverses the
    knowledge graph structure rather than finding similar text chunks.

    Examples of questions this mode handles that Modes 1 and 3 cannot:
        "Which models build on the Transformer architecture?"
        "What datasets are used to evaluate BERT?"
        "Which papers introduce methods that BERT improves on?"

    allow_dangerous_requests=True is required to enable write operations
    in the Cypher execution. For production read-only deployments, set
    allow_dangerous_requests=False and add a Cypher validation step.

    The validate_cypher=True parameter (available in LangChain 0.3+) enables
    syntax validation of the generated Cypher before execution, preventing
    malformed queries from reaching the database.

    Args:
        graph  : Connected Neo4jGraph with refreshed schema.
        llm    : AzureChatOpenAI.
        config : GraphRAGConfig.

    Returns:
        GraphCypherQAChain instance.
    """
    # Custom Cypher generation prompt that emphasizes schema adherence
    # and safe, read-only queries
    cypher_generation_prompt = PromptTemplate(
        input_variables=["schema", "question"],
        template="""You are a Neo4j Cypher expert. Generate a valid Cypher query
to answer the question using ONLY the schema provided.
Use only node labels and relationship types from the schema.
Return relevant properties (id, description, text) in the result.
Generate READ-ONLY queries only (MATCH, RETURN, WITH, WHERE, ORDER BY, LIMIT).
Do NOT use CREATE, MERGE, DELETE, SET.

Schema:
{schema}

Question: {question}

Cypher Query (no explanation, just the query):"""
    )

    chain = GraphCypherQAChain.from_llm(
        llm=llm,
        graph=graph,
        verbose=True,                           # logs the generated Cypher
        allow_dangerous_requests=True,
        validate_cypher=True,                   # syntax validation before execution
        cypher_prompt=cypher_generation_prompt,
        top_k=config.CYPHER_K,
        return_intermediate_steps=True,         # exposes generated Cypher for audit
    )

    logger.info("Mode 2 (Graph Cypher) chain assembled.")
    return chain


def build_mode3_hybrid_chain(
    vectorstore: Neo4jVector,
    llm: AzureChatOpenAI,
    config: GraphRAGConfig,
):
    """
    Mode 3: Hybrid retrieval with knowledge graph entity enrichment.

    This is the most powerful mode. It uses Neo4jVector with:
        search_type="hybrid": combines vector similarity + BM25 full-text search
        retrieval_query: a custom Cypher query that runs AFTER the initial
                         vector/keyword search to enrich each retrieved chunk
                         with its connected entities from the knowledge graph.

    The retrieval_query is the key innovation. After finding the top-k chunks
    by similarity/BM25, it executes:
        MATCH (node)-[:MENTIONS]->(entity)
        WITH node, score, collect(entity.id + " [" + labels(entity)[0] + "]") AS entities
        RETURN node.text, score, metadata{..., entities}

    This means every chunk the LLM receives is annotated with a list of the
    entities extracted from it during graph construction. The LLM sees not
    just "this text was semantically similar to your query" but also "this text
    discusses these specific entities: [Transformer, Multi-head Attention, BERT]".

    This structured entity context guides the LLM to produce more precise,
    entity-grounded answers and reduces hallucination by making the knowledge
    graph structure directly visible in the prompt context.

    Args:
        vectorstore : Neo4jVector with hybrid search_type and entity retrieval_query.
        llm         : AzureChatOpenAI.
        config      : GraphRAGConfig with HYBRID_RETRIEVAL_QUERY.

    Returns:
        Tuple of (LCEL chain, retriever).
    """
    # Rebuild the vectorstore with the custom retrieval query for entity enrichment
    hybrid_vectorstore = Neo4jVector(
        embedding=vectorstore.embedding,
        url=config.NEO4J_URI,
        username=config.NEO4J_USERNAME,
        password=config.NEO4J_PASSWORD,
        database=config.NEO4J_DATABASE,
        index_name=config.VECTOR_INDEX_NAME,
        keyword_index_name=config.KEYWORD_INDEX_NAME,
        search_type="hybrid",
        retrieval_query=config.HYBRID_RETRIEVAL_QUERY,
    )

    retriever = hybrid_vectorstore.as_retriever(
        search_kwargs={"k": config.HYBRID_K},
    )

    prompt = ChatPromptTemplate.from_template(config.RAG_PROMPT)

    def format_docs_with_entities(docs: List[Document]) -> str:
        """
        Format chunks with their extracted entity context.

        The custom retrieval_query puts the entities list in metadata['entities'].
        We render it as a structured annotation so the LLM can reference specific
        entities explicitly in its answer.
        """
        parts = []
        for i, doc in enumerate(docs, start=1):
            paper    = doc.metadata.get("paper_name", "Unknown")
            page     = doc.metadata.get("page", "?")
            entities = doc.metadata.get("entities", [])
            entity_str = (
                "  Entities: " + ", ".join(entities) if entities
                else "  Entities: (none extracted)"
            )
            parts.append(
                f"[Chunk {i} | {paper} | Page {page}]\n"
                f"{entity_str}\n\n"
                f"{doc.page_content.strip()}"
            )
        return ("\n\n" + "=" * 60 + "\n\n").join(parts)

    chain = (
        {
            "context":  retriever | format_docs_with_entities,
            "question": RunnablePassthrough(),
        }
        | prompt | llm | StrOutputParser()
    )

    logger.info("Mode 3 (Hybrid + Graph Entities) chain assembled.")
    return chain, retriever



In [85]:

# ===========================================================================
# SECTION 10 -- QUERY INTERFACE
# ===========================================================================

def query_graph_rag(
    mode1_chain,
    mode1_retriever,
    mode2_chain: GraphCypherQAChain,
    mode3_chain,
    mode3_retriever,
    question: str,
    modes: List[int] = [1, 2, 3],
) -> Dict[str, str]:
    """
    Execute a question through the selected Graph RAG retrieval modes.

    Modes can be selected independently per query based on the question type:
        Mode 1: Best for broad semantic questions ("Explain attention...")
        Mode 2: Best for relationship questions ("Which models build on X?")
        Mode 3: Best for entity-grounded factual questions (combines both)

    For each selected mode, prints retrieved sources/graph results and
    the LLM-generated answer.

    Args:
        mode1_chain      : Vector search chain.
        mode1_retriever  : Vector retriever for source display.
        mode2_chain      : GraphCypherQAChain for graph traversal.
        mode3_chain      : Hybrid chain.
        mode3_retriever  : Hybrid retriever for source display.
        question         : Natural language question string.
        modes            : List of mode integers [1, 2, 3] to run.

    Returns:
        Dict mapping mode label to answer string.
    """
    answers: Dict[str, str] = {}

    print("\n" + "#" * 72)
    print(f"QUESTION: {question}")
    print("#" * 72)

    # ---- Mode 1: Vector Search -------------------------------------------
    if 1 in modes:
        label = "Mode 1: Vector Search"
        print(f"\n--- {label} ---")
        source_docs = mode1_retriever.invoke(question)
        for i, doc in enumerate(source_docs, 1):
            paper = doc.metadata.get("paper_name", "?")
            page  = doc.metadata.get("page", "?")
            snip  = doc.page_content[:200].replace("\n", " ")
            print(f"  [{i}] {paper} | Page {page} | {snip}...")
        answer = mode1_chain.invoke(question)
        print(f"\nANSWER:\n{answer}")
        answers[label] = answer

    # ---- Mode 2: Graph Cypher -------------------------------------------
    if 2 in modes:
        label = "Mode 2: Graph Cypher Traversal"
        print(f"\n--- {label} ---")
        try:
            result = mode2_chain.invoke({"query": question})
            # result["result"] is the LLM answer
            # result["intermediate_steps"] contains the generated Cypher
            if "intermediate_steps" in result:
                for step in result["intermediate_steps"]:
                    if "query" in step:
                        print(f"  Generated Cypher:\n  {step['query']}")
            answer = result.get("result", "(no answer)")
            print(f"\nANSWER:\n{answer}")
            answers[label] = answer
        except Exception as exc:
            error_msg = f"Graph Cypher query failed: {exc}"
            logger.warning(error_msg)
            print(f"  WARNING: {error_msg}")
            answers[label] = error_msg

    # ---- Mode 3: Hybrid + Entity Enrichment ------------------------------
    if 3 in modes:
        label = "Mode 3: Hybrid + Knowledge Graph Entities"
        print(f"\n--- {label} ---")
        source_docs = mode3_retriever.invoke(question)
        for i, doc in enumerate(source_docs, 1):
            paper    = doc.metadata.get("paper_name", "?")
            page     = doc.metadata.get("page", "?")
            entities = doc.metadata.get("entities", [])
            snip     = doc.page_content[:200].replace("\n", " ")
            print(f"  [{i}] {paper} | Page {page} | Entities: {entities}")
            print(f"       {snip}...")
        answer = mode3_chain.invoke(question)
        print(f"\nANSWER:\n{answer}")
        answers[label] = answer

    return answers

In [86]:


# ===========================================================================
# SECTION 11 -- GRAPH INSPECTION UTILITIES
# Tools to inspect the knowledge graph state after construction
# ===========================================================================

def inspect_knowledge_graph(graph: Neo4jGraph) -> None:
    """
    Print a summary of the knowledge graph contents for verification.

    This diagnostic function is essential after ingestion to confirm:
    1. The correct number of entities were extracted.
    2. The relationship types match the ALLOWED_RELATIONSHIPS schema.
    3. The MENTIONS edges correctly link chunks to entities.

    Runs read-only Cypher queries against the live Neo4j instance.

    Args:
        graph: Connected Neo4jGraph instance.
    """
    print("\n" + "=" * 60)
    print("KNOWLEDGE GRAPH INSPECTION")
    print("=" * 60)

    # Node counts by label
    node_counts = graph.query(
        "MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count "
        "ORDER BY count DESC"
    )
    print("\nNode counts by label:")
    for row in node_counts:
        print(f"  {row['label']:<25} {row['count']:>6}")

    # Relationship type counts
    rel_counts = graph.query(
        "MATCH ()-[r]->() RETURN type(r) AS rel_type, count(r) AS count "
        "ORDER BY count DESC"
    )
    print("\nRelationship type counts:")
    for row in rel_counts:
        print(f"  {row['rel_type']:<30} {row['count']:>6}")

    # Sample entity nodes with their types
    sample_entities = graph.query(
        "MATCH (n:__Entity__) "
        "RETURN n.id AS entity, labels(n)[0] AS type "
        "ORDER BY entity LIMIT 20"
    )
    print("\nSample extracted entities:")
    for row in sample_entities:
        print(f"  [{row['type']:15}] {row['entity']}")

    # Sample relationships
    sample_rels = graph.query(
        "MATCH (a:__Entity__)-[r]->(b:__Entity__) "
        "RETURN a.id AS from_entity, type(r) AS rel, b.id AS to_entity "
        "LIMIT 15"
    )
    print("\nSample entity relationships:")
    for row in sample_rels:
        print(f"  {row['from_entity']} --[{row['rel']}]--> {row['to_entity']}")

    # Papers and their mention counts
    paper_mentions = graph.query(
        "MATCH (c:Chunk)-[:MENTIONS]->(e:__Entity__) "
        "RETURN c.paper_name AS paper, count(e) AS mentions "
        "ORDER BY mentions DESC"
    )
    print("\nEntity mentions by paper:")
    for row in paper_mentions:
        print(f"  {row['paper']:<45} {row['mentions']:>6} mentions")

    print("=" * 60 + "\n")



In [87]:

def check_graph_already_populated(graph: Neo4jGraph) -> bool:
    """
    Check if the Neo4j database already contains ingested data.

    Avoids re-running the expensive LLMGraphTransformer extraction on
    subsequent runs when the graph is already populated. Checks for
    the presence of (:Chunk) nodes as the canonical indicator.

    Args:
        graph: Connected Neo4jGraph instance.

    Returns:
        True if the database already contains Chunk nodes from ingestion.
    """
    result = graph.query("MATCH (n:Chunk) RETURN count(n) AS cnt")
    count = result[0]["cnt"]
    logger.info("Existing Chunk nodes in Neo4j: %d", count)
    return count > 0


In [101]:

# ===========================================================================
# SECTION 12 -- MAIN ORCHESTRATOR
# ===========================================================================

def main() -> None:
    """
    End-to-end Graph RAG pipeline orchestrator.

    Execution order:
        1.  Download three arXiv PDFs (cached after first run).
        2.  Load pages and split into 512-token chunks.
        3.  Initialize HuggingFace embedding model (local, no API key).
        4.  Validate and initialize Azure OpenAI LLM (early fail-fast).
        5.  Connect to Neo4j (validate connection with ping query).
        6.  Check if graph already populated (skip ingestion if so).
        7.  IF new ingestion:
              a. Extract knowledge graph via LLMGraphTransformer (async batch).
              b. Build Neo4j vector + keyword indexes on Chunk nodes.
        8.  Inspect knowledge graph contents (diagnostic report).
        9.  Build all three retrieval chains.
        10. Run demo queries across all three modes.

    On first run: steps 6-7b are executed (expensive -- ~5-15 min for 3 papers).
    On subsequent runs: steps 6-7b are skipped, chains load in seconds.

    Token cost estimate for graph extraction (step 7a):
        ~450 chunks * 512 input tokens/chunk * 2 (input+output)
        = ~460K tokens for all three papers
        At GPT-4o pricing ($5/1M input, $15/1M output):
        ~$2.50 USD total for one-time ingestion.
    """
    config = GraphRAGConfig()
    logger.info("=== Graph RAG Pipeline Starting ===")

    # Steps 1-2: Download and chunk documents
    pdf_paths = download_pdfs(config)
    chunks    = load_and_split_documents(pdf_paths, config)

    # Step 3: Embeddings
    embeddings = build_embeddings(config)

    # Step 4: LLM (validates Azure credentials before Neo4j connection)
    llm = build_azure_llm(config)

    # Step 5: Neo4j connection
    try:
        graph = connect_to_neo4j(config)
    except RuntimeError as exc:
        logger.error("Neo4j connection failed. Pipeline stopped before ingestion.")
        logger.error(str(exc))
        print("\nNeo4j connection failed. Please verify:")
        print("1. Aura instance is running")
        print("2. URI/username/password/database are correct")
        print("3. Your network/firewall allows outbound Neo4j TLS traffic")
        print("4. If using local Neo4j, set NEO4J_URI to bolt://localhost:7687")
        return

    # Step 6: Check if ingestion is needed
    already_populated = check_graph_already_populated(graph)

    if not already_populated:
        logger.info("Database empty. Running full ingestion pipeline ...")

        # Step 7a: Build knowledge graph (LLMGraphTransformer async extraction)
        build_knowledge_graph(chunks, llm, graph, config)

        # Step 7b: Build vector and keyword indexes on the ingested Chunk nodes
        vectorstore = build_vector_index(chunks, embeddings, graph, config)

        logger.info("Full ingestion complete.")
    else:
        logger.info("Database already populated. Loading existing indexes ...")

        # Connect to existing Neo4j vector index (no re-embedding)
        vectorstore = Neo4jVector(
            embedding=embeddings,
            url=config.NEO4J_URI,
            username=config.NEO4J_USERNAME,
            password=config.NEO4J_PASSWORD,
            database=config.NEO4J_DATABASE,
            index_name=config.VECTOR_INDEX_NAME,
            keyword_index_name=config.KEYWORD_INDEX_NAME,
            search_type="hybrid",
        )

    # Step 8: Inspect knowledge graph (diagnostic)
    inspect_knowledge_graph(graph)

    # Step 9: Build all three retrieval chains
    mode1_chain, mode1_retriever = build_mode1_vector_chain(vectorstore, llm, config)
    mode2_chain                  = build_mode2_graph_cypher_chain(graph, llm, config)
    mode3_chain, mode3_retriever = build_mode3_hybrid_chain(vectorstore, llm, config)

    # Step 10: Demo queries
    demo_queries = [
        # Mode 1 + 3 strength: semantic explanation
        (
            "What is the scaled dot-product attention mechanism and how does it work?",
            [1, 3],
        ),
        # Mode 2 strength: multi-hop graph relationship query
        (
            "Which model architectures build upon or improve on the original Transformer?",
            [2],
        ),
        # Mode 2 strength: entity relationship traversal
        (
            "What datasets are used to evaluate BERT's performance?",
            [2],
        ),
        # Mode 3 strength: entity-grounded factual question with hybrid recall
        (
            "How does BERT's masked language modeling objective differ from the Transformer's pre-training approach?",
            [1, 3],
        ),
        # Mode 2 strength: cross-paper relationship question impossible for standard RAG
        (
            "Which methods or concepts does the RAG model paper cite from prior work?",
            [2],
        ),
        # All three modes: comprehensive comparison
        (
            "What are the key contributions of each of the three papers in this corpus?",
            [1, 2, 3],
        ),
    ]

    logger.info("Running demo queries across all three retrieval modes ...")

    for question, modes in demo_queries:
        query_graph_rag(
            mode1_chain, mode1_retriever,
            mode2_chain,
            mode3_chain, mode3_retriever,
            question,
            modes=modes,
        )

    logger.info("=== Graph RAG Pipeline Demo Complete ===")



In [102]:

if __name__ == "__main__":
    main()


2026-04-08 09:20:59 | INFO     | graph_rag | === Graph RAG Pipeline Starting ===
2026-04-08 09:20:59 | INFO     | graph_rag | Cached: attention_is_all_you_need.pdf (2163.3 KB)
2026-04-08 09:20:59 | INFO     | graph_rag | Cached: bert_pretraining_transformers.pdf (757.0 KB)
2026-04-08 09:20:59 | INFO     | graph_rag | Cached: rag_knowledge_intensive_nlp.pdf (864.6 KB)
2026-04-08 09:20:59 | INFO     | graph_rag | Loading: attention_is_all_you_need.pdf
2026-04-08 09:21:00 | INFO     | graph_rag |   attention_is_all_you_need.pdf --> 15 pages --> 27 token chunks
2026-04-08 09:21:00 | INFO     | graph_rag | Loading: bert_pretraining_transformers.pdf
2026-04-08 09:21:01 | INFO     | graph_rag |   bert_pretraining_transformers.pdf --> 16 pages --> 46 token chunks
2026-04-08 09:21:01 | INFO     | graph_rag | Loading: rag_knowledge_intensive_nlp.pdf
2026-04-08 09:21:02 | INFO     | graph_rag |   rag_knowledge_intensive_nlp.pdf --> 19 pages --> 50 token chunks
2026-04-08 09:21:02 | INFO     | gra

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9083.73it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-04-08 09:21:04 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-08 09:21:04 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-04-08 09:21:05 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-08 09:21:05 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-08 09:21:05 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=fals